# CardioSignal: ECG ⇄ PCG training
Run with **Accelerator: GPU T4 x1 or P100** and the `cardiosignal-data` dataset attached.
Use **Save Version → Save & Run All** so it keeps running after you close the tab.

## 1. Locate data and code

In [ ]:
import glob, os, shutil, subprocess, sys
hits = glob.glob('/kaggle/input/**/train.npz', recursive=True)
assert hits, 'Dataset not found: attach the cardiosignal-data dataset (Add Input).'
DATA = os.path.dirname(hits[0])
CODE_SRC = glob.glob('/kaggle/input/**/code/train.py', recursive=True)[0].rsplit('/', 1)[0]
CODE = '/kaggle/working/code'
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
RUN = '/kaggle/working/run'
os.makedirs(RUN, exist_ok=True)
print('data:', DATA, os.listdir(DATA)); print('code:', CODE)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Resume support
If a previous run hit the time limit, add that notebook's output as an input; its `last.pt` is picked up here.

In [ ]:
prev = [p for p in glob.glob('/kaggle/input/**/run/last.pt', recursive=True)]
if prev and not os.path.exists(f'{RUN}/last.pt'):
    src = os.path.dirname(prev[0])
    for f in ('last.pt', 'best.pt', 'history.csv', 'best_val_metrics.json'):
        if os.path.exists(f'{src}/{f}'): shutil.copy(f'{src}/{f}', RUN)
    print('Resuming from', src)
else:
    print('Fresh run')

## 3. Tests + 1-minute smoke run (catches problems before the long run)

In [ ]:
%cd {CODE}
!python -m pytest -q -W ignore tests
!python -W ignore train.py --data_dir {DATA} --out_dir /kaggle/working/smoke --epochs 1 --max_train_batches 10 --max_val_batches 2 --batch_size 16
shutil.rmtree('/kaggle/working/smoke', ignore_errors=True)

## 4. Full training
Watch the `epoch N/60 ... min` lines to see how long each epoch takes.

In [ ]:
!python -W ignore train.py --data_dir {DATA} --out_dir {RUN} --epochs 60 --batch_size 32 --num_workers 2 --time_limit_hours 11

## 5. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(f'{RUN}/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
h.plot(x='epoch', y=['val_ecg_to_pcg_envelope_pearson', 'val_baseline_ecg_as_pcg_envelope_pearson'], ax=ax[0], title='ECG→PCG envelope correlation (val)')
h.plot(x='epoch', y=['val_pcg_to_ecg_pearson'], ax=ax[1], title='PCG→ECG correlation (val)')
plt.tight_layout(); plt.savefig(f'{RUN}/training_curves.png', dpi=110); plt.show()
h.tail()

## 6. Test-set evaluation (held-out subjects)
`final_ema.pt` (EMA weights after the last epoch) is the pre-chosen final model.

In [ ]:
!python -W ignore evaluate.py --checkpoint {RUN}/final_ema.pt --data_dir {DATA} --out_dir {RUN}/eval --num_plots 6
!python -W ignore beat_accuracy.py --checkpoint {RUN}/final_ema.pt --data_dir {DATA} --out {RUN}/eval/beat_accuracy.json
from IPython.display import Image, display
for p in sorted(glob.glob(f'{RUN}/eval/sample_*.png')): display(Image(p))

## 7. Package results
Download `results.zip` from the **Output** tab. `last.pt` stays in the output for resuming.

In [ ]:
%cd /kaggle/working
!zip -qr results.zip run/final_ema.pt run/best.pt run/history.csv run/best_val_metrics.json run/training_curves.png run/eval
!ls -lh results.zip run